# AutoChip Tutorial Assignment -- HW_AutoChip

**Course:** LLM4ChipDesign | **Student:** Anuj Apte

This notebook works through all eight AutoChip tutorial examples plus the
8-bit accumulator CPU stretch example, using AutoChip's real generate ->
compile (iverilog) -> simulate (vvp) -> feedback -> regenerate loop.

The AutoChip control-flow code in the next few cells (`Conversation`,
`LLMResponse`, `verilog_loop`, `compile_iverilog`, `find_verilog_modules`,
etc.) is copied essentially verbatim from
`AutoChip_Tutorial.ipynb` / the standalone
[AutoChip](https://github.com/shailja-thakur/AutoChip) package, so the loop
itself, the ranking rules, and the conversation bookkeeping are the real
AutoChip mechanics -- not a re-implementation.

**One adaptation was necessary and is called out explicitly below:** the
provided benchmark testbenches for these 8 examples are *self-checking*
(they `$display` an error and `$finish`, or print `All test cases passed!`)
rather than the VerilogEval-style testbenches AutoChip ships with, which
print a fixed `Mismatches: X in Y samples` line that the stock
`calculate_rank` regex looks for. `calculate_rank` below is adjusted to
parse pass/fail from the self-checking testbenches' own output instead,
while keeping the rest of the loop (conversation growth, iteration cap,
compile-error / simulate-error feedback messages) identical to AutoChip.

**How the "LLM" step works in this run:** each example's Verilog was
generated by the assignment author (using Claude) *acting as* the AutoChip
generation step -- i.e., manually playing the role `ChatGPT`/`Claude`
model classes normally play, reading the same conversation state AutoChip
would send an API, and returning a Verilog module in response. Real
`iverilog`/`vvp` tool feedback was then fed back into the next manual
attempt exactly as AutoChip's loop does automatically with a live API.
This is recorded faithfully below with a small `RecordedLLM` class that
replays those exact responses, iteration by iteration, so **this notebook
re-executes the real compile/simulate/feedback loop from scratch every time
it is run, with no API key required.** If you set `OPENAI_API_KEY` or
`ANTHROPIC_API_KEY` below and change `model_family`/`model_id` in a given
example's config to `"ChatGPT"` or `"Claude"`, that example will instead
call a live model through the unmodified AutoChip `ChatGPT`/`Claude`
classes and will very likely produce a different trajectory.


In [1]:
#@title Setting up the notebook
### Installing dependencies
!pip install -q openai tiktoken anthropic
!apt-get -qq update && apt-get -qq install -y iverilog


W: Failed to fetch https://download.docker.com/linux/ubuntu/dists/noble/InRelease  Invalid response from proxy: HTTP/1.1 403 Forbidden  Content-Type: text/plain; charset=utf-8  X-Content-Type-Options: nosniff  Content-Length: 77  Connection: close     [IP: 127.0.0.1 45041]
W: Some index files failed to download. They have been ignored, or old ones used instead.


In [2]:
#@title Utility functions (AutoChip core, with a self-checking-testbench rank adaptation)

import sys, os, re, json, subprocess
from abc import ABC, abstractmethod

################################################################################
### CONVERSATION CLASS  (verbatim from AutoChip's conversation.py)
################################################################################
class Conversation:
    def __init__(self, log_file=None):
        self.messages = []
        self.log_file = log_file
        if self.log_file and os.path.exists(self.log_file):
            open(self.log_file, 'w').close()

    def add_message(self, role, content):
        self.messages.append({'role': role, 'content': content})
        if self.log_file:
            with open(self.log_file, 'a') as file:
                file.write(f"{role}: {content}\n")

    def get_messages(self):
        return self.messages

    def remove_message(self, index):
        if index < len(self.messages):
            del self.messages[index]

################################################################################
### PARSING (verbatim from AutoChip's verilog_handling.py)
################################################################################
def find_verilog_modules(markdown_string):
    module_pattern = r'\bmodule\b\s+[\w\\_]+\s*(?:#\s*\([^)]*\))?\s*\([^)]*\)\s*;.*?endmodule\b'
    matches = re.findall(module_pattern, markdown_string, re.DOTALL)
    return [m.replace('\\_', '_') for m in matches]

def write_code_blocks_to_file(markdown_string, module_name, filename):
    code_match = find_verilog_modules(markdown_string)
    if not code_match:
        raise ValueError("No Verilog module found in response")
    with open(filename, 'w') as file:
        for block in code_match:
            file.write(block)
            file.write('\n')

def compile_iverilog(outdir, module, compiler_cmd, response):
    filename = os.path.join(outdir, module + ".sv")
    write_code_blocks_to_file(response.parsed_text, "module", filename)
    proc = subprocess.run(compiler_cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, timeout=120)
    return proc.returncode, proc.stderr, proc.stdout

def simulate_iverilog(simulation_cmd):
    proc = subprocess.run(simulation_cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, timeout=120)
    return proc.returncode, proc.stderr, proc.stdout

################################################################################
### LLMResponse -- calculate_rank ADAPTED for self-checking testbenches
### (everything else identical in spirit to AutoChip's LLMResponse)
################################################################################
class LLMResponse():
    def __init__(self, iteration, response_num, full_text):
        self.iteration = iteration
        self.response_num = response_num
        self.full_text = full_text
        self.parsed_text = ""
        self.parsed_length = 0
        self.feedback = ""
        self.compiled = False
        self.rank = -3
        self.message = ""

    def parse_verilog(self):
        module_list = find_verilog_modules(self.full_text)
        if not module_list:
            self.parsed_text = ""
        else:
            for module in module_list:
                self.parsed_text += module + "\n\n"
        self.parsed_length = len(self.parsed_text)

    def calculate_rank(self, outdir, module, testbench, tb_top):
        filename = os.path.join(outdir, module + ".sv")
        vvp_file = os.path.join(outdir, module + ".vvp")
        # -s <tb_top> instead of the AutoChip default "-s tb": these
        # self-checking testbenches name their top module explicitly
        # (e.g. tb_sequence_detector) rather than a bare "tb".
        compiler_cmd = f"iverilog -Wall -Winfloop -Wno-timescale -g2012 -s {tb_top} -o {vvp_file} {filename} {testbench}"
        simulator_cmd = f"vvp -n {vvp_file}"

        comp_return, comp_err, comp_out = compile_iverilog(outdir, module, compiler_cmd, self)

        if comp_return != 0:
            self.feedback = comp_err
            self.compiled = False
            self.message = "The design failed to compile. Please fix the module. The output of iverilog is as follows:\n" + comp_err
            self.rank = -1
            return

        sim_return, sim_err, sim_out = simulate_iverilog(simulator_cmd)
        full_out = sim_out + sim_err

        # --- adapted pass/fail detection for self-checking testbenches ---
        # AutoChip's stock rule looks for "Mismatches: X in Y samples"
        # (the VerilogEval convention). These testbenches instead print
        # "All test cases passed" / "Simulation successful" / "completed
        # successfully" on success, or one or more "Error: ..." lines on
        # failure.
        lower = full_out.lower()
        has_error_line = "error" in lower
        has_pass_phrase = any(p in lower for p in [
            "all test cases passed", "simulation successful",
            "completed successfully", "testbench completed successfully",
            "all cpu test cases passed", "result check passed",
        ])

        if has_pass_phrase and not has_error_line:
            self.compiled = True
            self.message = "The testbench completed successfully"
            self.rank = 1
        else:
            self.feedback = full_out
            self.compiled = True
            self.message = "The testbench simulated, but had errors. Please fix the module. The output of iverilog is as follows:\n" + full_out
            self.rank = 0

################################################################################
### RecordedLLM -- replays the author's real per-iteration responses
### (drop-in replacement for AutoChip's ChatGPT/Claude/... classes)
################################################################################
class AbstractLLM(ABC):
    @abstractmethod
    def generate(self, conversation, num_candidates=1):
        pass

class RecordedLLM(AbstractLLM):
    """Replays a fixed list of previously-authored responses, one per
    iteration, so the notebook is 100% reproducible without an API key."""
    def __init__(self, responses):
        self._responses = responses
        self._n = 0

    def generate(self, conversation, num_candidates=1):
        text = self._responses[min(self._n, len(self._responses) - 1)]
        self._n += 1
        return [text]

import openai
class ChatGPT(AbstractLLM):
    """Unmodified AutoChip OpenAI backend -- used only if you set
    OPENAI_API_KEY and select model_family='ChatGPT' below."""
    def __init__(self, model_id="gpt-4o-mini"):
        openai.api_key = os.environ.get('OPENAI_API_KEY', '')
        self.client = openai.OpenAI()
        self.model_id = model_id

    def generate(self, conversation, num_candidates=1):
        messages = [{"role": m["role"], "content": m["content"]} for m in conversation.get_messages()]
        response = self.client.chat.completions.create(model=self.model_id, n=num_candidates, messages=messages)
        return [c.message.content for c in response.choices]

################################################################################
### verilog_loop -- structurally identical to AutoChip's, generalized to
### take a tb_top name (see calculate_rank note above) and an injected
### LLM instance instead of re-instantiating one from a family string.
################################################################################
def verilog_loop(design_prompt, module, testbench, tb_top, max_iterations, llm, outdir="", log=None):
    if outdir != "":
        outdir = outdir + "/"
        os.makedirs(outdir, exist_ok=True)

    conv = Conversation(log_file=log)
    conv.add_message("system", (
        "You are an autocomplete engine for Verilog code. "
        "Given a Verilog module specification, you will provide a completed Verilog module in response. "
        "You will provide completed Verilog modules for all specifications, and will not create any supplementary modules. "
        "Given a Verilog module that is either incorrect/compilation error, you will suggest corrections to the module. "
        "You will not refuse. "
        "Format your response as Verilog code containing the end to end corrected module and not just the corrected lines inside ``` tags, do not include anything else inside ``` ."
    ))
    conv.add_message("user", design_prompt)

    success = False
    iterations = 0
    global_max_response = LLMResponse(-3, -3, "")

    while not success and iterations <= max_iterations:
        response_texts = llm.generate(conv)
        responses = [LLMResponse(iterations, i, t) for i, t in enumerate(response_texts)]

        for index, response in enumerate(responses):
            response_outdir = os.path.join(outdir, f"iter{iterations}/response{index}/")
            os.makedirs(response_outdir, exist_ok=True)
            response.parse_verilog()
            if response.parsed_text == "":
                response.rank = -2
                response.message = "No modules found in response"
            else:
                response.calculate_rank(response_outdir, module, testbench, tb_top)
            print(f"  iter {iterations}: compiled={response.compiled} rank={response.rank}")

        max_rank_response = max(responses, key=lambda r: (r.rank, -r.parsed_length))
        if max_rank_response.rank > global_max_response.rank:
            global_max_response = max_rank_response

        conv.add_message("assistant", max_rank_response.parsed_text)

        if max_rank_response.rank == 1:
            success = True
        else:
            if iterations > 0:
                conv.remove_message(2)
                conv.remove_message(2)
            conv.add_message("user", max_rank_response.message)

        iterations += 1

    return global_max_response, conv


## Setting the API key

In [3]:
### API keys (optional -- see the note in the intro cell above).
### This run uses RecordedLLM and needs neither key. Uncomment and fill
### these in, and pass model_family="ChatGPT" instead of a RecordedLLM
### instance to an example below, to call a live model instead.

# os.environ["OPENAI_API_KEY"] = ""
# os.environ["ANTHROPIC_API_KEY"] = ""


# Example 1: binary_to_bcd (combinational converter)

**Note on this example's trajectory:** Combinational converter; iverilog compiled cleanly and the self-checking testbench (32 exhaustive cases) passed on the first generated attempt.

In [4]:
#@title Setting up files: binary_to_bcd
%cd /content
!mkdir -p binary_to_bcd_example
design_prompt = "I am trying to create a Verilog model for a binary to binary-coded-decimal converter. It must meet the following specifications:\n\t- Inputs:\n\t\t- Binary input (5-bits)\n\t- Outputs:\n\t\t- BCD (8-bits: 4-bits for the 10's place and 4-bits for the 1's place)\n\nHow would I write a design that meets these specifications?"
testbench_text = "`timescale 1ns / 1ps\n\nmodule tb_binary_to_bcd_converter;\n\nreg [4:0] binary_input;\nwire [7:0] bcd_output;\n\nbinary_to_bcd_converter uut (\n    .binary_input(binary_input),\n    .bcd_output(bcd_output)\n);\n\ninteger i;\nreg [4:0] test_binary;\nreg [7:0] expected_bcd;\n\ninitial begin\n    $display(\"Testing Binary-to-BCD Converter...\");\n\n    for (i = 0; i < 32; i++) begin\n        test_binary = i;\n        binary_input = test_binary;\n\n        // Calculate expected BCD output\n        expected_bcd[3:0] = test_binary % 10;\n        expected_bcd[7:4] = test_binary / 10;\n\n        #10; // Wait for the results\n\n        if (bcd_output !== expected_bcd) begin\n            $display(\"Error: Test case %0d failed. Expected BCD: 8'b%0b, Got: 8'b%0b\",\n                     test_binary, expected_bcd, bcd_output);\n            $finish;\n        end\n    end\n\n    $display(\"All test cases passed!\");\n    $finish;\nend\n\nreg vcd_clk;\ninitial begin\n    $dumpfile(\"my_design.vcd\");\n    $dumpvars(0, tb_binary_to_bcd_converter);\nend\n\nalways #5 vcd_clk = ~vcd_clk; // Toggle clock every 5 time units\n\nendmodule\n\n"
with open("binary_to_bcd_example/binary_to_bcd.prompt.md", "w") as f:
    f.write(design_prompt)
with open("binary_to_bcd_example/binary_to_bcd_tb.v", "w") as f:
    f.write(testbench_text)
print(design_prompt)


/content


I am trying to create a Verilog model for a binary to binary-coded-decimal converter. It must meet the following specifications:
	- Inputs:
		- Binary input (5-bits)
	- Outputs:
		- BCD (8-bits: 4-bits for the 10's place and 4-bits for the 1's place)

How would I write a design that meets these specifications?


In [5]:
%cd /content/binary_to_bcd_example
recorded_responses = ["```verilog\nmodule binary_to_bcd_converter (\n    input  [4:0] binary_input,\n    output reg [7:0] bcd_output\n);\n    always @(*) begin\n        bcd_output[7:4] = binary_input / 10;\n        bcd_output[3:0] = binary_input % 10;\n    end\nendmodule\n\n```"]
llm = RecordedLLM(recorded_responses)

best, conv = verilog_loop(
    design_prompt=design_prompt,
    module="binary_to_bcd_converter",
    testbench="binary_to_bcd_tb.v",
    tb_top="tb_binary_to_bcd_converter",
    max_iterations=0,
    llm=llm,
    outdir="run_out",
    log="log.txt",
)
print("\nFinal rank:", best.rank, " (1 = testbench passed)")
print("\nFinal module:\n")
print(best.parsed_text)
%cd /content


/content/binary_to_bcd_example
  iter 0: compiled=True rank=1

Final rank: 1  (1 = testbench passed)

Final module:

module binary_to_bcd_converter (
    input  [4:0] binary_input,
    output reg [7:0] bcd_output
);
    always @(*) begin
        bcd_output[7:4] = binary_input / 10;
        bcd_output[3:0] = binary_input % 10;
    end
endmodule


/content


# Example 2: sequence_detector (FSM sequence matching)

**Note on this example's trajectory:** First attempt registered `sequence_found` from a fully-registered 8-symbol shift history; iverilog compiled it cleanly but simulation failed at the final symbol because the testbench samples the output in the same simulation step as the clock edge, before the DUT's nonblocking updates settle -- so a purely registered comparison is always read one cycle stale. Iteration 2 fixed this by keeping only the previous 7 symbols in a register and comparing them combinationally against the live `data` input plus the target pattern, which removed the extra register stage feeding the checked output. This is the example highlighted for the report as showing a meaningful, feedback-driven RTL change.

In [6]:
#@title Setting up files: sequence_detector
%cd /content
!mkdir -p sequence_detector_example
design_prompt = "I am trying to create a Verilog model for a sequence detector. It must meet the following specifications:\n\t- Inputs:\n\t\t- Clock\n\t\t- Active-low reset\n\t\t- Data (3 bits)\n\t- Outputs:\n\t\t- Sequence found\n\nWhile enabled, it should detect the following sequence of binary input values:\n\t- 0b001\n\t- 0b101\n\t- 0b110\n\t- 0b000\n\t- 0b110\n\t- 0b110\n\t- 0b011\n\t- 0b101\n\nHow would I write a design that meets these specifications?"
testbench_text = "`timescale 1ns/1ps\n\nmodule tb_sequence_detector();\n    reg clk;\n    reg reset_n;\n    reg [2:0] data;\n    wire sequence_found;\n\n    // Instantiate the sequence_detector module\n    sequence_detector dut (\n        .clk(clk),\n        .reset_n(reset_n),\n        .data(data),\n        .sequence_found(sequence_found)\n    );\n\n    // Clock generation\n    always begin\n        #5 clk = ~clk;\n    end\n\n    // Test stimulus task\n    task apply_stimulus;\n        input [2:0] data_value;\n        input integer delay_cycles;\n        begin\n            data <= data_value;\n            repeat (delay_cycles) @(posedge clk);\n        end\n    endtask\n\n    // Check output task\n    task check_output;\n        input integer cycle;\n        input expected_value;\n        begin\n            if (sequence_found !== expected_value) begin\n                $display(\"Error: Cycle %0d, Expected: %b, Got: %b\", cycle, expected_value, sequence_found);\n                $finish;\n            end\n        end\n    endtask\n\n    // Testbench stimulus and checking\n    initial begin\n        // Initialize signals\n        clk <= 0;\n        reset_n <= 0;\n        data <= 3'b000;\n\n        // Apply reset\n        @(posedge clk);\n        reset_n <= 1;\n\n        // Test case: Correct sequence\n        apply_stimulus(3'b001, 1); check_output(1, 1'b0);\n        apply_stimulus(3'b101, 1); check_output(2, 1'b0);\n        apply_stimulus(3'b110, 1); check_output(3, 1'b0);\n        apply_stimulus(3'b000, 1); check_output(4, 1'b0);\n        apply_stimulus(3'b110, 1); check_output(5, 1'b0);\n        apply_stimulus(3'b110, 1); check_output(6, 1'b0);\n        apply_stimulus(3'b011, 1); check_output(7, 1'b0);\n        apply_stimulus(3'b101, 1); check_output(8, 1'b1);\n\n        // Test case: Incorrect sequence\n        apply_stimulus(3'b001, 1); check_output(9, 1'b0);\n        apply_stimulus(3'b101, 1); check_output(10, 1'b0);\n        apply_stimulus(3'b010, 1); check_output(11, 1'b0);\n        apply_stimulus(3'b000, 1); check_output(12, 1'b0);\n\n        // Indicate successful test completion\n        $display(\"All test cases passed.\");\n        $finish;\n    end\n\nendmodule\n\n"
with open("sequence_detector_example/sequence_detector.prompt.md", "w") as f:
    f.write(design_prompt)
with open("sequence_detector_example/sequence_detector_tb.v", "w") as f:
    f.write(testbench_text)
print(design_prompt)


/content


I am trying to create a Verilog model for a sequence detector. It must meet the following specifications:
	- Inputs:
		- Clock
		- Active-low reset
		- Data (3 bits)
	- Outputs:
		- Sequence found

While enabled, it should detect the following sequence of binary input values:
	- 0b001
	- 0b101
	- 0b110
	- 0b000
	- 0b110
	- 0b110
	- 0b011
	- 0b101

How would I write a design that meets these specifications?


In [7]:
%cd /content/sequence_detector_example
recorded_responses = ["```verilog\nmodule sequence_detector (\n    input clk,\n    input reset_n,\n    input [2:0] data,\n    output reg sequence_found\n);\n    reg [23:0] history;\n    localparam [23:0] TARGET = {3'b001,3'b101,3'b110,3'b000,3'b110,3'b110,3'b011,3'b101};\n\n    always @(posedge clk or negedge reset_n) begin\n        if (!reset_n) begin\n            history <= 24'b0;\n            sequence_found <= 1'b0;\n        end else begin\n            history <= {history[20:0], data};\n            sequence_found <= ({history[20:0], data} == TARGET);\n        end\n    end\nendmodule\n\n```", "```verilog\nmodule sequence_detector (\n    input clk,\n    input reset_n,\n    input [2:0] data,\n    output sequence_found\n);\n    // History holds the previous 7 symbols; the testbench samples\n    // sequence_found combinationally right after asserting the new\n    // symbol and before the next clock edge's registers have settled,\n    // so the comparison must be combinational (current data + the\n    // already-registered previous 7 symbols) rather than fully\n    // registered, to avoid a one-cycle lag against the checker.\n    reg [20:0] history;\n    localparam [23:0] TARGET = {3'b001,3'b101,3'b110,3'b000,3'b110,3'b110,3'b011,3'b101};\n\n    always @(posedge clk or negedge reset_n) begin\n        if (!reset_n)\n            history <= 21'b0;\n        else\n            history <= {history[17:0], data};\n    end\n\n    assign sequence_found = ({history, data} == TARGET);\nendmodule\n\n```"]
llm = RecordedLLM(recorded_responses)

best, conv = verilog_loop(
    design_prompt=design_prompt,
    module="sequence_detector",
    testbench="sequence_detector_tb.v",
    tb_top="tb_sequence_detector",
    max_iterations=1,
    llm=llm,
    outdir="run_out",
    log="log.txt",
)
print("\nFinal rank:", best.rank, " (1 = testbench passed)")
print("\nFinal module:\n")
print(best.parsed_text)
%cd /content


/content/sequence_detector_example
  iter 0: compiled=True rank=0
  iter 1: compiled=True rank=1

Final rank: 1  (1 = testbench passed)

Final module:

module sequence_detector (
    input clk,
    input reset_n,
    input [2:0] data,
    output sequence_found
);
    // History holds the previous 7 symbols; the testbench samples
    // sequence_found combinationally right after asserting the new
    // symbol and before the next clock edge's registers have settled,
    // so the comparison must be combinational (current data + the
    // already-registered previous 7 symbols) rather than fully
    // registered, to avoid a one-cycle lag against the checker.
    reg [20:0] history;
    localparam [23:0] TARGET = {3'b001,3'b101,3'b110,3'b000,3'b110,3'b110,3'b011,3'b101};

    always @(posedge clk or negedge reset_n) begin
        if (!reset_n)
            history <= 21'b0;
        else
            history <= {history[17:0], data};
    end

    assign sequence_found = ({history, data} == 

# Example 3: shift_register (sequential shift register)

**Note on this example's trajectory:** First attempt left data_out undefined (X) because the testbench never actually pulses reset_n low before its first check; iteration 2 added a simulation-time initial value. The testbench still fails: its expected-output constant (test_case_data_out) does not depend on data_in/shift_enable at all -- every checked value equals 10 right-shifted by the loop index, which we verified directly against Icarus rather than against our own arithmetic. No design that actually implements the specified shift register (or any alternative shift direction / enable interpretation we tried) can reproduce that input-independent sequence, so this looks like a bug in the provided testbench's reference data rather than in the DUT. This is documented as the remaining, unresolved issue for this example.

In [8]:
#@title Setting up files: shift_register
%cd /content
!mkdir -p shift_register_example
design_prompt = "I am trying to create a Verilog model for a shift register. It must meet the following specifications:\n\t- Inputs:\n\t\t- Clock\n\t\t- Active-low reset\n\t\t- Data (1 bit)\n\t\t- Shift enable\n\t- Outputs:\n\t\t- Data (8 bits)\n\nHow would I write a design that meets these specifications?"
testbench_text = "`timescale 1ns/1ps\n\nmodule tb_shift_register();\n    reg clk;\n    reg reset_n;\n    reg data_in;\n    reg shift_enable;\n    wire [7:0] data_out;\n\n    // Instantiate the shift_register\n    shift_register dut (\n        .clk(clk),\n        .reset_n(reset_n),\n        .data_in(data_in),\n        .shift_enable(shift_enable),\n        .data_out(data_out)\n    );\n\n    // Clock generation\n    always begin\n        #5 clk = ~clk;\n    end\n\n    // Test case data\n    reg [7:0] test_case_reset_n = 8'b00111111;\n    reg [7:0] test_case_data_in = 8'b01010100;\n    reg [7:0] test_case_shift_enable = 8'b00111010;\n    reg [63:0] test_case_data_out = 64'b00000000000000000000000000000001000000100000010100001010;\n\n    integer i;\n\n    // Test runner\n    initial begin\n        clk = 1;\n        reset_n = 1;\n        data_in = 0;\n        shift_enable = 0;\n\n        for (i = 0; i < 7; i = i + 1) begin\n            reset_n <= test_case_reset_n[i];\n            data_in <= test_case_data_in[i];\n            shift_enable <= test_case_shift_enable[i];\n            @(posedge clk);\n\n            if (data_out !== test_case_data_out[8*i+:8]) begin\n                $display(\"Error: Test case %0d failed. Expected: %b, Got: %b\", i, test_case_data_out[8*i+:8], data_out);\n                $finish;\n            end\n        end\n\n        $display(\"All test cases passed!\");\n        $finish;\n    end\nendmodule\n\n"
with open("shift_register_example/shift_register.prompt.md", "w") as f:
    f.write(design_prompt)
with open("shift_register_example/shift_register_tb.v", "w") as f:
    f.write(testbench_text)
print(design_prompt)


/content


I am trying to create a Verilog model for a shift register. It must meet the following specifications:
	- Inputs:
		- Clock
		- Active-low reset
		- Data (1 bit)
		- Shift enable
	- Outputs:
		- Data (8 bits)

How would I write a design that meets these specifications?


In [9]:
%cd /content/shift_register_example
recorded_responses = ["```verilog\nmodule shift_register (\n    input clk,\n    input reset_n,\n    input data_in,\n    input shift_enable,\n    output reg [7:0] data_out\n);\n    always @(posedge clk or negedge reset_n) begin\n        if (!reset_n)\n            data_out <= 8'b0;\n        else if (shift_enable)\n            data_out <= {data_out[6:0], data_in};\n    end\nendmodule\n\n```", "```verilog\nmodule shift_register (\n    input clk,\n    input reset_n,\n    input data_in,\n    input shift_enable,\n    output reg [7:0] data_out\n);\n    // Give data_out a defined value at t=0 in addition to the\n    // asynchronous active-low reset, since the testbench never\n    // pulses reset_n low before its first check.\n    initial data_out = 8'b0;\n\n    always @(posedge clk or negedge reset_n) begin\n        if (!reset_n)\n            data_out <= 8'b0;\n        else if (shift_enable)\n            data_out <= {data_out[6:0], data_in};\n    end\nendmodule\n\n```"]
llm = RecordedLLM(recorded_responses)

best, conv = verilog_loop(
    design_prompt=design_prompt,
    module="shift_register",
    testbench="shift_register_tb.v",
    tb_top="tb_shift_register",
    max_iterations=1,
    llm=llm,
    outdir="run_out",
    log="log.txt",
)
print("\nFinal rank:", best.rank, " (1 = testbench passed)")
print("\nFinal module:\n")
print(best.parsed_text)
%cd /content


/content/shift_register_example


  iter 0: compiled=True rank=0


  iter 1: compiled=True rank=0

Final rank: 0  (1 = testbench passed)

Final module:

module shift_register (
    input clk,
    input reset_n,
    input data_in,
    input shift_enable,
    output reg [7:0] data_out
);
    always @(posedge clk or negedge reset_n) begin
        if (!reset_n)
            data_out <= 8'b0;
        else if (shift_enable)
            data_out <= {data_out[6:0], data_in};
    end
endmodule


/content


# Example 4: abro_state_machine (ABRO handshake style FSM)

**Note on this example's trajectory:** One-hot ABRO FSM with an explicit hold-state for the A=B=0 case; compiled and passed all 16 testbench cases on the first attempt.

In [10]:
#@title Setting up files: abro_state_machine
%cd /content
!mkdir -p abro_state_machine_example
design_prompt = "I am trying to create a Verilog model for an ABRO state machine. It must meet the following specifications:\n    - Inputs:\n        - Clock\n        - Active-low reset\n        - A\n        - B\n    - Outputs:\n        - O\n        - State\n\nOther than the main output from ABRO machine, it should output the current state of the machine for use in verification.\n\nThe states for this state machine should be one-hot encoded.\n\nHow would I write a design that meets these specifications?"
testbench_text = "`timescale 1ns/1ps\n\nmodule tb_abro_state_machine();\n    reg clk;\n    reg rst_n;\n    reg A;\n    reg B;\n    wire O;\n    wire [3:0] State;\n\n    // Instantiate the ABRO state machine\n    abro_state_machine uut (\n        .clk(clk),\n        .rst_n(rst_n),\n        .A(A),\n        .B(B),\n        .O(O),\n        .State(State)\n    );\n\n    // Clock generation\n    always begin\n        #5 clk = ~clk;\n    end\n\n    integer i;\n    reg [3:0] expected_state;\n    reg expected_O;\n\n    // Test stimulus and checking\n    initial begin\n        // Initialize signals\n        clk = 0;\n        rst_n = 1;\n        A = 0;\n        B = 0;\n\n        // Test cases\n        for (i = 0; i < 16; i = i + 1) begin\n            // Determine the input and expected output based on the test case index\n            case (i)\n                0: begin\n                    A = 0; B = 0; // Reset\n                    expected_O = 1'b0;\n                    expected_state = 4'b0001;\n                end\n                1: begin\n                    A = 0; B = 1; // A=0, B=1\n                    expected_O = 1'b0;\n                    expected_state = 4'b0001;\n                end\n                2: begin\n                    A = 1; B = 0; // A=1, B=0\n                    expected_O = 1'b0;\n                    expected_state = 4'b0010;\n                end\n                3: begin\n                    A = 1; B = 1; // A=1, B=1\n                    expected_O = 1'b1;\n                    expected_state = 4'b0100;\n                end\n                default: begin\n                    // Repeat the same sequence of inputs for the remaining test cases\n                    A = (i-1) % 2;\n                    B = ((i-1) / 2) % 2;\n                    expected_O = (A && B) ? 1'b1 : 1'b0;\n                    expected_state = (A && !B) ? 4'b0010 : ((!A && B) ? 4'b0001 : 4'b0100);\n                end\n            endcase\n\n            // Apply reset\n            if (i == 0) begin\n                rst_n = 0;\n                @(posedge clk);\n                rst_n = 1;\n            end\n\n            @(posedge clk); // Wait for the state change on the rising edge of the clock\n\n            // Check the output and state\n            if (O !== expected_O || State !== expected_state) begin\n                $display(\"Error: Test case %0d failed (A=%b, B=%b)\", i, A, B);\n                $display(\"  Expected: O=%b, State=%b\", expected_O, expected_state);\n                $display(\"  Got: O=%b, State=%b\", O, State);\n                $finish;\n            end\n        end\n\n        $display(\"All test cases passed.\");\n        $finish;\n    end\n\n\n\treg vcd_clk;\ninitial begin\n    $dumpfile(\"my_design.vcd\");\n    $dumpvars(0, tb_abro_state_machine);\nend\n\nalways #5 vcd_clk = ~vcd_clk; // Toggle clock every 5 time units\n\nendmodule\n\n"
with open("abro_state_machine_example/abro_state_machine.prompt.md", "w") as f:
    f.write(design_prompt)
with open("abro_state_machine_example/abro_state_machine_tb.v", "w") as f:
    f.write(testbench_text)
print(design_prompt)


/content


I am trying to create a Verilog model for an ABRO state machine. It must meet the following specifications:
    - Inputs:
        - Clock
        - Active-low reset
        - A
        - B
    - Outputs:
        - O
        - State

Other than the main output from ABRO machine, it should output the current state of the machine for use in verification.

The states for this state machine should be one-hot encoded.

How would I write a design that meets these specifications?


In [11]:
%cd /content/abro_state_machine_example
recorded_responses = ["```verilog\nmodule abro_state_machine (\n    input clk,\n    input rst_n,\n    input A,\n    input B,\n    output reg O,\n    output reg [3:0] State\n);\n    always @(posedge clk or negedge rst_n) begin\n        if (!rst_n) begin\n            State <= 4'b0001;\n            O     <= 1'b0;\n        end else begin\n            O <= (A && B) ? 1'b1 : 1'b0;\n            if (A && !B)\n                State <= 4'b0010;\n            else if (!A && B)\n                State <= 4'b0001;\n            else if (A && B)\n                State <= 4'b0100;\n            // else A==0 && B==0: hold current State\n        end\n    end\nendmodule\n\n```"]
llm = RecordedLLM(recorded_responses)

best, conv = verilog_loop(
    design_prompt=design_prompt,
    module="abro_state_machine",
    testbench="abro_state_machine_tb.v",
    tb_top="tb_abro_state_machine",
    max_iterations=0,
    llm=llm,
    outdir="run_out",
    log="log.txt",
)
print("\nFinal rank:", best.rank, " (1 = testbench passed)")
print("\nFinal module:\n")
print(best.parsed_text)
%cd /content


/content/abro_state_machine_example
  iter 0: compiled=True rank=1

Final rank: 1  (1 = testbench passed)

Final module:

module abro_state_machine (
    input clk,
    input rst_n,
    input A,
    input B,
    output reg O,
    output reg [3:0] State
);
    always @(posedge clk or negedge rst_n) begin
        if (!rst_n) begin
            State <= 4'b0001;
            O     <= 1'b0;
        end else begin
            O <= (A && B) ? 1'b1 : 1'b0;
            if (A && !B)
                State <= 4'b0010;
            else if (!A && B)
                State <= 4'b0001;
            else if (A && B)
                State <= 4'b0100;
            // else A==0 && B==0: hold current State
        end
    end
endmodule


/content


# Example 5: dice_roller (randomized behavior)

**Note on this example's trajectory:** Free-running 8-bit Fibonacci LFSR reduced modulo the selected die's side count; passed the 4x1000-roll range/statistics check on the first attempt.

In [12]:
#@title Setting up files: dice_roller
%cd /content
!mkdir -p dice_roller_example
design_prompt = "I am trying to create a Verilog model for a simulated dice roller. It must meet the following specifications:\n    - Inputs:\n        - Clock\n        - Active-low reset\n        - Die select (2-bits)\n        - Roll\n    - Outputs:\n        - Rolled number (up to 8-bits)\n\nThe design should simulate rolling either a 4-sided, 6-sided, 8-sided, or 20-sided die, based on the input die select. It should roll when the roll input goes high and output the random number based on the number of sides of the selected die.\n\nHow would I write a design that meets these specifications?"
testbench_text = "`timescale 1ns / 1ps\n\nmodule tb_dice_roller();\n    reg clk;\n    reg rst_n;\n    reg [1:0] die_select;\n    reg roll;\n    wire [7:0] rolled_number;\n\n    dice_roller dut (\n        .clk(clk),\n        .rst_n(rst_n),\n        .die_select(die_select),\n        .roll(roll),\n        .rolled_number(rolled_number)\n    );\n\n    // Clock generation\n    always begin\n        #5 clk = ~clk;\n    end\n\n    integer i;\n    integer j;\n    integer error_count;\n    reg [31:0] roll_counts [0:20]; // Declare roll_counts as a memory\n\n    // Testbench stimulus\n    initial begin\n\n        clk = 0;\n        rst_n = 0;\n        die_select = 0;\n        roll = 0;\n\n        // Reset and initialization\n        #10 rst_n = 1;\n        #10 roll = 1;\n\n        error_count = 0;\n\n        // Test loop\n        for (i = 0; i < 4; i++) begin\n            die_select = i;\n\n            // Clear roll_counts\n            for (j = 1; j <= 20; j++) begin\n                roll_counts[j] = 0;\n            end\n\n            // Perform 1000 rolls and count the results\n            for (j = 0; j < 1000; j++) begin\n                #10;\n                roll = 0;\n                #10;\n                roll = 1;\n                #10;\n                roll = 0;\n                #10;\n\n                // Check the rolled_number is within the expected range\n                case (die_select)\n                    2'b00: begin\n                        if (rolled_number < 1 || rolled_number > 4) begin\n                            $display(\"Error: Invalid roll result for 4-sided die: %d\", rolled_number);\n                            error_count = error_count + 1;\n                        end\n                    end\n                    2'b01: begin\n                        if (rolled_number < 1 || rolled_number > 6) begin\n                            $display(\"Error: Invalid roll result for 6-sided die: %d\", rolled_number);\n                            error_count = error_count + 1;\n                        end\n                    end\n                    2'b10: begin\n                        if (rolled_number < 1 || rolled_number > 8) begin\n                            $display(\"Error: Invalid roll result for 8-sided die: %d\", rolled_number);\n                            error_count = error_count + 1;\n                        end\n                    end\n                    2'b11: begin\n                        if (rolled_number < 1 || rolled_number > 20) begin\n                            $display(\"Error: Invalid roll result for 20-sided die: %d\", rolled_number);\n                            error_count = error_count + 1;\n                        end\n                    end\n                endcase\n\n                roll_counts[rolled_number] = roll_counts[rolled_number] + 1;\n            end\n\n            $display(\"Results for die_select %b:\", die_select);\n            for (j = 1; j <= 20; j++) begin\n                if (roll_counts[j] > 0) begin\n                    $display(\"  Rolled %d: %d times\", j, roll_counts[j]);\n                end\n            end\n        end\n\n        if (error_count == 0) begin\n            $display(\"Testbench completed successfully.\");\n        end else begin\n            $display(\"Testbench completed with %d errors.\", error_count);\n        end\n\n        $finish;\n    end\n\nreg vcd_clk;\ninitial begin\n    $dumpfile(\"my_design.vcd\");\n    $dumpvars(0, tb_dice_roller);\nend\n\nalways #5 vcd_clk = ~vcd_clk; // Toggle clock every 5 time units\nendmodule\n\n"
with open("dice_roller_example/dice_roller.prompt.md", "w") as f:
    f.write(design_prompt)
with open("dice_roller_example/dice_roller_tb.v", "w") as f:
    f.write(testbench_text)
print(design_prompt)


/content


I am trying to create a Verilog model for a simulated dice roller. It must meet the following specifications:
    - Inputs:
        - Clock
        - Active-low reset
        - Die select (2-bits)
        - Roll
    - Outputs:
        - Rolled number (up to 8-bits)

The design should simulate rolling either a 4-sided, 6-sided, 8-sided, or 20-sided die, based on the input die select. It should roll when the roll input goes high and output the random number based on the number of sides of the selected die.

How would I write a design that meets these specifications?


In [13]:
%cd /content/dice_roller_example
recorded_responses = ["```verilog\nmodule dice_roller (\n    input clk,\n    input rst_n,\n    input [1:0] die_select,\n    input roll,\n    output reg [7:0] rolled_number\n);\n    reg [7:0] lfsr;\n    wire feedback = lfsr[7] ^ lfsr[5] ^ lfsr[4] ^ lfsr[3];\n\n    always @(posedge clk or negedge rst_n) begin\n        if (!rst_n)\n            lfsr <= 8'hA5;\n        else\n            lfsr <= {lfsr[6:0], feedback};\n    end\n\n    reg [4:0] sides;\n    always @(*) begin\n        case (die_select)\n            2'b00: sides = 5'd4;\n            2'b01: sides = 5'd6;\n            2'b10: sides = 5'd8;\n            default: sides = 5'd20;\n        endcase\n    end\n\n    always @(posedge clk or negedge rst_n) begin\n        if (!rst_n)\n            rolled_number <= 8'd1;\n        else if (roll)\n            rolled_number <= (lfsr % sides) + 1'b1;\n    end\nendmodule\n\n```"]
llm = RecordedLLM(recorded_responses)

best, conv = verilog_loop(
    design_prompt=design_prompt,
    module="dice_roller",
    testbench="dice_roller_tb.v",
    tb_top="tb_dice_roller",
    max_iterations=0,
    llm=llm,
    outdir="run_out",
    log="log.txt",
)
print("\nFinal rank:", best.rank, " (1 = testbench passed)")
print("\nFinal module:\n")
print(best.parsed_text)
%cd /content


/content/dice_roller_example


  iter 0: compiled=True rank=1

Final rank: 1  (1 = testbench passed)

Final module:

module dice_roller (
    input clk,
    input rst_n,
    input [1:0] die_select,
    input roll,
    output reg [7:0] rolled_number
);
    reg [7:0] lfsr;
    wire feedback = lfsr[7] ^ lfsr[5] ^ lfsr[4] ^ lfsr[3];

    always @(posedge clk or negedge rst_n) begin
        if (!rst_n)
            lfsr <= 8'hA5;
        else
            lfsr <= {lfsr[6:0], feedback};
    end

    reg [4:0] sides;
    always @(*) begin
        case (die_select)
            2'b00: sides = 5'd4;
            2'b01: sides = 5'd6;
            2'b10: sides = 5'd8;
            default: sides = 5'd20;
        endcase
    end

    always @(posedge clk or negedge rst_n) begin
        if (!rst_n)
            rolled_number <= 8'd1;
        else if (roll)
            rolled_number <= (lfsr % sides) + 1'b1;
    end
endmodule


/content


# Example 6: lfsr (linear feedback shift register)

**Note on this example's trajectory:** Direct translation of the tap/seed spec; matched the testbench's bit-exact 256-cycle reference sequence on the first attempt.

In [14]:
#@title Setting up files: lfsr
%cd /content
!mkdir -p lfsr_example
design_prompt = "I am trying to create a Verilog model for an LFSR. It must meet the following specifications:\n\t- Inputs:\n\t\t- Clock\n        - Active-low reset\n\t- Outputs:\n\t\t- Data (8-bits)\n\nThe initial state should be 10001010, and the taps should be at locations 1, 4, 6, and 7.\n\nHow would I write a design that meets these specifications?"
testbench_text = "`timescale 1ns / 1ps\n\nmodule tb_lfsr();\n    reg clk;\n    reg reset_n;\n    wire [7:0] data;\n\n    // Instantiate the LFSR design\n    lfsr lfsr_inst (\n        .clk(clk),\n        .reset_n(reset_n),\n        .data(data)\n    );\n\n    // Clock generation\n    always begin\n        #5 clk = ~clk;\n    end\n\n    // Variables for stimulus and checking\n    integer i;\n    reg [7:0] data_expected;\n\n    // Function to generate the reference LFSR output\n    function [7:0] reference_lfsr_output;\n        input [7:0] current_state;\n        begin\n            reference_lfsr_output = {current_state[6:0], current_state[0] ^ current_state[3] ^ current_state[5] ^ current_state[6]};\n        end\n    endfunction\n\n    // Stimulus and checking\n    initial begin\n        // Initialize the clock and reset signals\n        clk = 0;\n        reset_n = 0;\n\n        // Apply reset and wait for a clock cycle\n        #5 reset_n = 1;\n\n        // Initial state check\n        if (data !== 8'b10001010) begin\n            $display(\"Error: At time %t, data = %b, expected = 10001010\", $time, data);\n            $finish;\n        end\n\n        // Set the initial expected state\n        data_expected = 8'b10001010;\n\n        // Run simulation for 256 cycles and check the output\n        for (i = 0; i < 256; i = i + 1) begin\n            // Update the expected state\n            data_expected = reference_lfsr_output(data_expected);\n\n            // Apply the clock\n            #10;\n\n            // Check the output\n            if (data !== data_expected) begin\n                $display(\"Error: At time %t, data = %b, expected = %b\", $time, data, data_expected);\n                $finish;\n            end\n        end\n\n        // Simulation successful\n        $display(\"Simulation successful. All test cases passed.\");\n        $finish;\n    end\n\nendmodule\n\n"
with open("lfsr_example/lfsr.prompt.md", "w") as f:
    f.write(design_prompt)
with open("lfsr_example/lfsr_tb.v", "w") as f:
    f.write(testbench_text)
print(design_prompt)


/content


I am trying to create a Verilog model for an LFSR. It must meet the following specifications:
	- Inputs:
		- Clock
        - Active-low reset
	- Outputs:
		- Data (8-bits)

The initial state should be 10001010, and the taps should be at locations 1, 4, 6, and 7.

How would I write a design that meets these specifications?


In [15]:
%cd /content/lfsr_example
recorded_responses = ["```verilog\nmodule lfsr (\n    input  clk,\n    input  reset_n,\n    output reg [7:0] data\n);\n    wire feedback = data[0] ^ data[3] ^ data[5] ^ data[6];\n\n    always @(posedge clk or negedge reset_n) begin\n        if (!reset_n)\n            data <= 8'b10001010;\n        else\n            data <= {data[6:0], feedback};\n    end\nendmodule\n\n```"]
llm = RecordedLLM(recorded_responses)

best, conv = verilog_loop(
    design_prompt=design_prompt,
    module="lfsr",
    testbench="lfsr_tb.v",
    tb_top="tb_lfsr",
    max_iterations=0,
    llm=llm,
    outdir="run_out",
    log="log.txt",
)
print("\nFinal rank:", best.rank, " (1 = testbench passed)")
print("\nFinal module:\n")
print(best.parsed_text)
%cd /content


/content/lfsr_example
  iter 0: compiled=True rank=1

Final rank: 1  (1 = testbench passed)

Final module:

module lfsr (
    input  clk,
    input  reset_n,
    output reg [7:0] data
);
    wire feedback = data[0] ^ data[3] ^ data[5] ^ data[6];

    always @(posedge clk or negedge reset_n) begin
        if (!reset_n)
            data <= 8'b10001010;
        else
            data <= {data[6:0], feedback};
    end
endmodule


/content


# Example 7: sequence_generator (pattern generator)

**Note on this example's trajectory:** First attempt registered `data` directly (index then output both delayed by clocked registers); the testbench again samples immediately after the posedge, before the DUT's own nonblocking updates for that edge settle, so it saw the previous symbol. Iteration 2 made `data` a continuous read of `seq[idx]`, with only `idx` registered, which resolved the same class of race seen in sequence_detector and passed generation, repetition, and disable checks.

In [16]:
#@title Setting up files: sequence_generator
%cd /content
!mkdir -p sequence_generator_example
design_prompt = "I am trying to create a Verilog model for a sequence generator. It must meet the following specifications:\n\t- Inputs:\n\t\t- Clock\n\t\t- Active-low reset\n\t\t- Enable\n\t- Outputs:\n\t\t- Data (8 bits)\n\nWhile enabled, it should generate an output sequence of the following hexadecimal values and then repeat:\n\t- 0xAF\n\t- 0xBC\n\t- 0xE2\n\t- 0x78\n\t- 0xFF\n\t- 0xE2\n\t- 0x0B\n\t- 0x8D\n\nHow would I write a design that meets these specifications?"
testbench_text = "`timescale 1ns/1ps\n\nmodule tb_sequence_generator();\n    reg clk;\n    reg reset_n;\n    reg enable;\n    wire [7:0] data;\n\n    // Instantiate the sequence_generator\n    sequence_generator seq_gen (\n        .clk(clk),\n        .reset_n(reset_n),\n        .enable(enable),\n        .data(data)\n    );\n\n    // Clock generation\n    always begin\n        #5 clk = ~clk;\n    end\n\n    // Test sequence\n    localparam TEST_SEQ_LEN = 8;\n    reg [7:0] test_sequence [0:TEST_SEQ_LEN-1];\n\n    // Initialize test sequence array\n    initial begin\n        test_sequence[0] = 8'hAF;\n        test_sequence[1] = 8'hBC;\n        test_sequence[2] = 8'hE2;\n        test_sequence[3] = 8'h78;\n        test_sequence[4] = 8'hFF;\n        test_sequence[5] = 8'hE2;\n        test_sequence[6] = 8'h0B;\n        test_sequence[7] = 8'h8D;\n    end\n\n    integer i;\n\n    // Testbench\n    initial begin\n        // Initialize signals\n        clk = 0;\n        reset_n = 0;\n        enable = 0;\n\n        // Apply reset\n        #10 reset_n = 1;\n        #10 reset_n = 0;\n        #10 reset_n = 1;\n\n        // Test case: sequence generation\n        enable = 1;\n        for (i = 0; i < TEST_SEQ_LEN; i = i + 1) begin\n            @(posedge clk);\n            if (data !== test_sequence[i]) begin\n                $display(\"Error: Mismatch at position %0d. Expected: %h, Got: %h\", i, test_sequence[i], data);\n                $finish;\n            end\n        end\n        $display(\"Test case passed: sequence generation\");\n\n        // Test case: sequence repetition\n        for (i = 0; i < TEST_SEQ_LEN; i = i + 1) begin\n            @(posedge clk);\n            if (data !== test_sequence[i]) begin\n                $display(\"Error: Mismatch in sequence repetition at position %0d. Expected: %h, Got: %h\", i, test_sequence[i], data);\n                $finish;\n            end\n        end\n        $display(\"Test case passed: sequence repetition\");\n\n        // Test case: disable the sequence generator\n        enable = 0;\n        @(posedge clk);\n        if (data !== test_sequence[TEST_SEQ_LEN-1]) begin\n            $display(\"Error: Sequence generator not disabled. Expected: %h, Got: %h\", test_sequence[TEST_SEQ_LEN-1], data);\n            $finish;\n        end\n        $display(\"Test case passed: disable sequence generator\");\n\n        $display(\"All test cases passed\");\n        $finish;\n    end\n\nendmodule\n\n"
with open("sequence_generator_example/sequence_generator.prompt.md", "w") as f:
    f.write(design_prompt)
with open("sequence_generator_example/sequence_generator_tb.v", "w") as f:
    f.write(testbench_text)
print(design_prompt)


/content


I am trying to create a Verilog model for a sequence generator. It must meet the following specifications:
	- Inputs:
		- Clock
		- Active-low reset
		- Enable
	- Outputs:
		- Data (8 bits)

While enabled, it should generate an output sequence of the following hexadecimal values and then repeat:
	- 0xAF
	- 0xBC
	- 0xE2
	- 0x78
	- 0xFF
	- 0xE2
	- 0x0B
	- 0x8D

How would I write a design that meets these specifications?


In [17]:
%cd /content/sequence_generator_example
recorded_responses = ["```verilog\nmodule sequence_generator (\n    input clk,\n    input reset_n,\n    input enable,\n    output reg [7:0] data\n);\n    reg [2:0] idx;\n    reg [7:0] seq [0:7];\n\n    initial begin\n        seq[0] = 8'hAF;\n        seq[1] = 8'hBC;\n        seq[2] = 8'hE2;\n        seq[3] = 8'h78;\n        seq[4] = 8'hFF;\n        seq[5] = 8'hE2;\n        seq[6] = 8'h0B;\n        seq[7] = 8'h8D;\n    end\n\n    always @(posedge clk or negedge reset_n) begin\n        if (!reset_n) begin\n            idx  <= 3'd0;\n            data <= seq[0];\n        end else if (enable) begin\n            data <= seq[idx];\n            idx  <= idx + 1'b1;\n        end\n    end\nendmodule\n\n```", "```verilog\nmodule sequence_generator (\n    input clk,\n    input reset_n,\n    input enable,\n    output [7:0] data\n);\n    // The testbench samples `data` immediately after each posedge,\n    // before this edge's own nonblocking updates have settled, so\n    // the output must be combinational off of a register that was\n    // already settled as of the *previous* edge (same pattern as\n    // sequence_detector), rather than itself carrying a one-cycle\n    // register delay.\n    reg [2:0] idx;\n    reg [7:0] seq [0:7];\n\n    initial begin\n        seq[0] = 8'hAF;\n        seq[1] = 8'hBC;\n        seq[2] = 8'hE2;\n        seq[3] = 8'h78;\n        seq[4] = 8'hFF;\n        seq[5] = 8'hE2;\n        seq[6] = 8'h0B;\n        seq[7] = 8'h8D;\n    end\n\n    assign data = seq[idx];\n\n    always @(posedge clk or negedge reset_n) begin\n        if (!reset_n)\n            idx <= 3'd0;\n        else if (enable)\n            idx <= idx + 1'b1;\n    end\nendmodule\n\n```"]
llm = RecordedLLM(recorded_responses)

best, conv = verilog_loop(
    design_prompt=design_prompt,
    module="sequence_generator",
    testbench="sequence_generator_tb.v",
    tb_top="tb_sequence_generator",
    max_iterations=1,
    llm=llm,
    outdir="run_out",
    log="log.txt",
)
print("\nFinal rank:", best.rank, " (1 = testbench passed)")
print("\nFinal module:\n")
print(best.parsed_text)
%cd /content


/content/sequence_generator_example
  iter 0: compiled=True rank=0
  iter 1: compiled=True rank=1

Final rank: 1  (1 = testbench passed)

Final module:

module sequence_generator (
    input clk,
    input reset_n,
    input enable,
    output [7:0] data
);
    // The testbench samples `data` immediately after each posedge,
    // before this edge's own nonblocking updates have settled, so
    // the output must be combinational off of a register that was
    // already settled as of the *previous* edge (same pattern as
    // sequence_detector), rather than itself carrying a one-cycle
    // register delay.
    reg [2:0] idx;
    reg [7:0] seq [0:7];

    initial begin
        seq[0] = 8'hAF;
        seq[1] = 8'hBC;
        seq[2] = 8'hE2;
        seq[3] = 8'h78;
        seq[4] = 8'hFF;
        seq[5] = 8'hE2;
        seq[6] = 8'h0B;
        seq[7] = 8'h8D;
    end

    assign data = seq[idx];

    always @(posedge clk or negedge reset_n) begin
        if (!reset_n)
            idx <=

# Example 8: traffic_light (traffic light FSM)

**Note on this example's trajectory:** 3-state counter-based FSM (RED/GREEN/YELLOW) with the specified 32/20/7 cycle counts; passed all 4 testbench checks on the first attempt.

In [18]:
#@title Setting up files: traffic_light
%cd /content
!mkdir -p traffic_light_example
design_prompt = "I am trying to create a Verilog model for a traffic light state machine. It must meet the following specifications:\n    - Inputs:\n        - Clock\n        - Active-low reset\n        - Enable\n    - Outputs:\n        - Red\n        - Yellow\n        - Green\n\nThe state machine should reset to a red light, change from red to green after 32 clock cycles, change from green to yellow after 20 clock cycles, and then change from yellow to red after 7 clock cycles.\n\nHow would I write a design that meets these specifications?"
testbench_text = "`timescale 1ps/1ps\n\nmodule tb_traffic_light_fsm();\n\nreg clk;\nreg reset_n;\nreg enable;\nwire red;\nwire yellow;\nwire green;\n\ninteger error_count;\n\n// Instantiate the traffic_light_fsm\ntraffic_light_fsm traffic_light (\n    .clk(clk),\n    .reset_n(reset_n),\n    .enable(enable),\n    .red(red),\n    .yellow(yellow),\n    .green(green)\n);\n\n// Clock generation\nalways begin\n    #5 clk = ~clk;\nend\n\n// Check the output and print test results\ntask check_output;\n    input integer cycle;\n    input logic exp_red;\n    input logic exp_yellow;\n    input logic exp_green;\n    input integer test_case;\nbegin\n    @(negedge clk);\n    if (red !== exp_red || yellow !== exp_yellow || green !== exp_green) begin\n        $display(\"Error: Test case %0d failed - FSM did not transition to %s after %0d clock cycles.\", test_case, exp_red ? \"RED\" : (exp_yellow ? \"YELLOW\" : \"GREEN\"), cycle);\n        error_count = error_count + 1;\n    end else begin\n        $display(\"Test case %0d passed - FSM transitioned to %s after %0d clock cycles.\", test_case, exp_red ? \"RED\" : (exp_yellow ? \"YELLOW\" : \"GREEN\"), cycle);\n    end\nend\nendtask\n\ninitial begin\n    // Initialize signals\n    clk = 0;\n    reset_n = 0;\n    enable = 1;\n\n    // Initialize error_count\n    error_count = 0;\n\n    // Test case 1 - Check if the FSM starts with the RED state\n    check_output(0, 1, 0, 0, 1);\n\n    // Apply reset\n    #5 reset_n = 1;\n\n    // Test case 2 - Check if the FSM transitions to GREEN after 32 clock cycles\n    repeat (32) @(posedge clk);\n    check_output(32, 0, 0, 1, 2);\n\n    // Test case 3 - Check if the FSM transitions to YELLOW after 20 clock cycles\n    repeat (20) @(posedge clk);\n    check_output(20, 0, 1, 0, 3);\n\n    // Test case 4 - Check if the FSM transitions to RED after 7 clock cycles\n    repeat (7) @(posedge clk);\n    check_output(7, 1, 0, 0, 4);\n\n    if (error_count > 0) begin\n        $display(\"Error: %0d test cases failed.\", error_count);\n    end else begin\n        $display(\"All test cases passed!\");\n    end\n    $finish;\nend\n\nendmodule\n\n"
with open("traffic_light_example/traffic_light.prompt.md", "w") as f:
    f.write(design_prompt)
with open("traffic_light_example/traffic_light_tb.v", "w") as f:
    f.write(testbench_text)
print(design_prompt)


/content


I am trying to create a Verilog model for a traffic light state machine. It must meet the following specifications:
    - Inputs:
        - Clock
        - Active-low reset
        - Enable
    - Outputs:
        - Red
        - Yellow
        - Green

The state machine should reset to a red light, change from red to green after 32 clock cycles, change from green to yellow after 20 clock cycles, and then change from yellow to red after 7 clock cycles.

How would I write a design that meets these specifications?


In [19]:
%cd /content/traffic_light_example
recorded_responses = ["```verilog\nmodule traffic_light_fsm (\n    input clk,\n    input reset_n,\n    input enable,\n    output reg red,\n    output reg yellow,\n    output reg green\n);\n    localparam S_RED = 2'b00, S_GREEN = 2'b01, S_YELLOW = 2'b10;\n    reg [1:0] state;\n    reg [5:0] count;\n\n    always @(posedge clk or negedge reset_n) begin\n        if (!reset_n) begin\n            state <= S_RED;\n            count <= 6'd0;\n        end else if (enable) begin\n            case (state)\n                S_RED:    if (count == 6'd31) begin state <= S_GREEN;  count <= 0; end\n                          else count <= count + 1'b1;\n                S_GREEN:  if (count == 6'd19) begin state <= S_YELLOW; count <= 0; end\n                          else count <= count + 1'b1;\n                S_YELLOW: if (count == 6'd6)  begin state <= S_RED;    count <= 0; end\n                          else count <= count + 1'b1;\n                default:  state <= S_RED;\n            endcase\n        end\n    end\n\n    always @(*) begin\n        red = 1'b0; yellow = 1'b0; green = 1'b0;\n        case (state)\n            S_RED:    red    = 1'b1;\n            S_GREEN:  green  = 1'b1;\n            S_YELLOW: yellow = 1'b1;\n        endcase\n    end\nendmodule\n\n```"]
llm = RecordedLLM(recorded_responses)

best, conv = verilog_loop(
    design_prompt=design_prompt,
    module="traffic_light_fsm",
    testbench="traffic_light_tb.v",
    tb_top="tb_traffic_light_fsm",
    max_iterations=0,
    llm=llm,
    outdir="run_out",
    log="log.txt",
)
print("\nFinal rank:", best.rank, " (1 = testbench passed)")
print("\nFinal module:\n")
print(best.parsed_text)
%cd /content


/content/traffic_light_example
  iter 0: compiled=True rank=1

Final rank: 1  (1 = testbench passed)

Final module:

module traffic_light_fsm (
    input clk,
    input reset_n,
    input enable,
    output reg red,
    output reg yellow,
    output reg green
);
    localparam S_RED = 2'b00, S_GREEN = 2'b01, S_YELLOW = 2'b10;
    reg [1:0] state;
    reg [5:0] count;

    always @(posedge clk or negedge reset_n) begin
        if (!reset_n) begin
            state <= S_RED;
            count <= 6'd0;
        end else if (enable) begin
            case (state)
                S_RED:    if (count == 6'd31) begin state <= S_GREEN;  count <= 0; end
                          else count <= count + 1'b1;
                S_GREEN:  if (count == 6'd19) begin state <= S_YELLOW; count <= 0; end
                          else count <= count + 1'b1;
                S_YELLOW: if (count == 6'd6)  begin state <= S_RED;    count <= 0; end
                          else count <= count + 1'b1;
            

# Part II -- 8-bit CPU example

**Starting co-design prompt (Fig. 10 in the assignment PDF):**

> Let us make a brand new microprocessor design together. We're severely
> constrained on space and I/O. We have to fit in 1000 standard cells of
> an ASIC, so I think we will need to restrict ourselves to an
> accumulator based 8-bit architecture with no multi-byte instructions.
> Given this, how do you think we should begin?

This is explored as a short **co-design** (not a single-shot AutoChip
generate/compile/verify loop like the 8 examples above): the trajectory
below is the sequence of architecture decisions worked through with the
model, each grounded against the stated constraint, before a single RTL
implementation was written and verified against a testbench with
`iverilog`/`vvp`.

**Trajectory:**
1. *Instruction width* -- "no multi-byte instructions" forces exactly one
   byte per instruction, so opcode and operand must share 8 bits.
2. *Opcode/operand split* -- a 3-bit opcode (8 opcodes) leaves 5 operand
   bits (32 addressable words), which comfortably covers a small program +
   data footprint for a ~1000-cell budget.
3. *ISA* -- settled on the minimal accumulator set needed for real
   programs: `NOP, LDA, STA, ADD, SUB, JMP, JZ, HALT`.
4. *Datapath* -- Harvard split (separate 32x8 program ROM addressed by a
   5-bit PC, and 32x8 data RAM addressed by the operand field) rather than
   a shared bus, since it keeps the control logic single-cycle and simple
   for the cell budget.
5. *Control* -- single always block, single-cycle fetch+execute per
   instruction (no separate fetch/decode/execute pipeline stages, again to
   stay small).
6. *RTL + verification* -- implemented the above and verified it against a
   testbench that loads a short program summing 5+4+3+2+1 via a
   decrement loop and checks the final accumulator/memory state and that
   the CPU halts.


In [20]:
#@title Setting up files: cpu_8bit
%cd /content
!mkdir -p cpu_8bit_example
cpu_prompt = "Let us make a brand new microprocessor design together. We're severely constrained on space and I/O. We have to fit in 1000 standard cells of an ASIC, so I think we will need to restrict ourselves to an accumulator based 8-bit architecture with no multi-byte instructions. Given this, how do you think we should begin?"
with open("cpu_8bit_example/cpu_8bit_tb.v", "w") as f:
    f.write("\n`timescale 1ns/1ps\n// Testbench for the 8-bit accumulator CPU.\n// Program (program.hex) sums the integers 5,4,3,2,1 using a decrement loop:\n//   data_mem[0] = counter (starts at 5)\n//   data_mem[1] = running sum (starts at 0)\n//   data_mem[2] = constant 1 (for decrementing the counter)\n//   data_mem[3] = result (written by the program, expected = 15)\nmodule tb_cpu_8bit;\n    reg clk;\n    reg reset_n;\n    wire [7:0] acc;\n    wire [4:0] pc;\n    wire halted;\n\n    cpu_8bit dut (\n        .clk(clk),\n        .reset_n(reset_n),\n        .acc(acc),\n        .pc(pc),\n        .halted(halted)\n    );\n\n    always #5 clk = ~clk;\n\n    integer cycles;\n    integer errors;\n\n    initial begin\n        clk = 0;\n        reset_n = 0;\n        errors = 0;\n\n        // Seed data memory (simulates loading initial data alongside the program)\n        dut.data_mem[0] = 8'd5;  // counter\n        dut.data_mem[1] = 8'd0;  // sum\n        dut.data_mem[2] = 8'd1;  // constant 1\n        dut.data_mem[3] = 8'd0;  // result (to be written)\n\n        @(negedge clk);\n        reset_n = 1;\n\n        cycles = 0;\n        while (!halted && cycles < 200) begin\n            @(negedge clk);\n            cycles = cycles + 1;\n        end\n\n        if (!halted) begin\n            $display(\"Error: CPU did not halt within %0d cycles\", cycles);\n            errors = errors + 1;\n        end else begin\n            $display(\"CPU halted after %0d cycles. PC=%0d ACC=%0d\", cycles, pc, acc);\n        end\n\n        if (dut.data_mem[3] !== 8'd15) begin\n            $display(\"Error: expected data_mem[3] = 15 (5+4+3+2+1), got %0d\", dut.data_mem[3]);\n            errors = errors + 1;\n        end else begin\n            $display(\"Result check passed: data_mem[3] = %0d\", dut.data_mem[3]);\n        end\n\n        if (dut.data_mem[0] !== 8'd0) begin\n            $display(\"Error: expected counter (data_mem[0]) to reach 0, got %0d\", dut.data_mem[0]);\n            errors = errors + 1;\n        end\n\n        if (errors == 0)\n            $display(\"All CPU test cases passed!\");\n        else\n            $display(\"CPU testbench completed with %0d error(s).\", errors);\n\n        $finish;\n    end\nendmodule\n")
with open("cpu_8bit_example/program.hex", "w") as f:
    f.write("\n20\nC8\n61\n41\n20\n82\n40\nA0\n21\n43\nE0\n00\n00\n00\n00\n00\n00\n00\n00\n00\n00\n00\n00\n00\n00\n00\n00\n00\n00\n00\n00\n00\n")
print(cpu_prompt)


/content


Let us make a brand new microprocessor design together. We're severely constrained on space and I/O. We have to fit in 1000 standard cells of an ASIC, so I think we will need to restrict ourselves to an accumulator based 8-bit architecture with no multi-byte instructions. Given this, how do you think we should begin?


In [21]:
%cd /content/cpu_8bit_example
cpu_response = "```verilog\n// 8-bit accumulator-based microprocessor.\n// Co-design constraints from the starting prompt: severely limited\n// space/IO (~1000 standard cells), accumulator architecture, no\n// multi-byte instructions -> every instruction is exactly one byte.\n//\n// Instruction format: [7:5] = opcode (3 bits), [4:0] = operand/address (5 bits)\n//   000 NOP\n//   001 LDA addr   ACC <= DMEM[addr]\n//   010 STA addr   DMEM[addr] <= ACC\n//   011 ADD addr   ACC <= ACC + DMEM[addr]\n//   100 SUB addr   ACC <= ACC - DMEM[addr]\n//   101 JMP addr   PC  <= addr\n//   110 JZ  addr   if (ACC==0) PC <= addr; else PC <= PC + 1\n//   111 HALT       halted <= 1\n//\n// Harvard split: 32x8 program ROM addressed by 5-bit PC, 32x8 data RAM\n// addressed by the 5-bit operand. Single-cycle fetch+execute per instruction.\n\nmodule cpu_8bit (\n    input  wire       clk,\n    input  wire       reset_n,\n    output reg  [7:0] acc,\n    output reg  [4:0] pc,\n    output reg        halted\n);\n    localparam OP_NOP  = 3'b000;\n    localparam OP_LDA  = 3'b001;\n    localparam OP_STA  = 3'b010;\n    localparam OP_ADD  = 3'b011;\n    localparam OP_SUB  = 3'b100;\n    localparam OP_JMP  = 3'b101;\n    localparam OP_JZ   = 3'b110;\n    localparam OP_HALT = 3'b111;\n\n    reg [7:0] prog_mem [0:31];\n    reg [7:0] data_mem [0:31];\n\n    initial $readmemh(\"program.hex\", prog_mem);\n\n    wire [7:0] instr   = prog_mem[pc];\n    wire [2:0] opcode  = instr[7:5];\n    wire [4:0] operand = instr[4:0];\n\n    always @(posedge clk) begin\n        if (!reset_n) begin\n            acc    <= 8'd0;\n            pc     <= 5'd0;\n            halted <= 1'b0;\n        end else if (!halted) begin\n            case (opcode)\n                OP_NOP:  pc <= pc + 5'd1;\n                OP_LDA:  begin acc <= data_mem[operand]; pc <= pc + 5'd1; end\n                OP_STA:  begin data_mem[operand] <= acc; pc <= pc + 5'd1; end\n                OP_ADD:  begin acc <= acc + data_mem[operand]; pc <= pc + 5'd1; end\n                OP_SUB:  begin acc <= acc - data_mem[operand]; pc <= pc + 5'd1; end\n                OP_JMP:  pc <= operand;\n                OP_JZ:   pc <= (acc == 8'd0) ? operand : pc + 5'd1;\n                OP_HALT: halted <= 1'b1;\n                default: pc <= pc + 5'd1;\n            endcase\n        end\n    end\nendmodule\n\n```"
llm = RecordedLLM([cpu_response])

best, conv = verilog_loop(
    design_prompt=cpu_prompt,
    module="cpu_8bit",
    testbench="cpu_8bit_tb.v",
    tb_top="tb_cpu_8bit",
    max_iterations=0,
    llm=llm,
    outdir="run_out",
    log="log.txt",
)
print("\nFinal rank:", best.rank, " (1 = testbench passed)")
print("\nFinal module:\n")
print(best.parsed_text)
%cd /content


/content/cpu_8bit_example
  iter 0: compiled=True rank=1

Final rank: 1  (1 = testbench passed)

Final module:

module cpu_8bit (
    input  wire       clk,
    input  wire       reset_n,
    output reg  [7:0] acc,
    output reg  [4:0] pc,
    output reg        halted
);
    localparam OP_NOP  = 3'b000;
    localparam OP_LDA  = 3'b001;
    localparam OP_STA  = 3'b010;
    localparam OP_ADD  = 3'b011;
    localparam OP_SUB  = 3'b100;
    localparam OP_JMP  = 3'b101;
    localparam OP_JZ   = 3'b110;
    localparam OP_HALT = 3'b111;

    reg [7:0] prog_mem [0:31];
    reg [7:0] data_mem [0:31];

    initial $readmemh("program.hex", prog_mem);

    wire [7:0] instr   = prog_mem[pc];
    wire [2:0] opcode  = instr[7:5];
    wire [4:0] operand = instr[4:0];

    always @(posedge clk) begin
        if (!reset_n) begin
            acc    <= 8'd0;
            pc     <= 5'd0;
            halted <= 1'b0;
        end else if (!halted) begin
            case (opcode)
                OP_NOP:  pc <=

**Result / discussion:**

The single-cycle accumulator CPU above compiled and passed its testbench
(halts after the expected number of cycles with the correct accumulator
and memory result) without needing an RTL repair iteration -- the
iteration effort for this example went into the *architecture* decisions
in the trajectory above rather than into fixing failed RTL.

- **How AutoChip-style decomposition mapped onto this design:** the ISA,
  datapath and control questions were resolved in a fixed order (word
  width -> opcode/operand split -> instruction set -> memory
  organization -> control style) because each later decision depends on
  the earlier ones; this is a much larger decomposition than any of the 8
  tutorial examples need.
- **Does compiler/simulator feedback still work at this scale?** Yes for
  RTL-level bugs (a wrong case statement or off-by-one would show up in
  iverilog/vvp output exactly as in the smaller examples), but it cannot
  catch an *architectural* mistake (e.g. too few operand bits to address
  the needed memory) -- that has to be caught by reasoning about the
  constraint before RTL is written, not by tool feedback afterward.
- **Hardest part to automate:** the constraint-driven ISA design step.
  Picking the opcode/operand split and instruction set that fit "1000
  standard cells" and "no multi-byte instructions" is a judgment call with
  no compiler feedback signal at all -- iverilog has no opinion on whether
  8 opcodes is the right number, only on whether the Verilog you already
  wrote for the ISA you already chose compiles and simulates correctly.
- **Remaining limitations:** no interrupts or stack (so no subroutine
  calls), fixed 32-word program/data memories, undefined behavior on PC
  overflow past address 31, and no realistic timing/area analysis against
  the actual 1000-cell target (only cycle-accurate functional
  verification was performed here).
